# 📘 GUÍA DEL PROFESOR — Unidad 4: Proyecto Integrador
## Pipeline Multiagente: Análisis y Predicción de Ventas en Retail
**TecNM Campus Apatzingán | Ingeniería en Sistemas Computacionales**

> 🔒 **Uso exclusivo del instructor.**
> Contiene el código de referencia completo, los resultados esperados y las respuestas
> a cada pregunta guía para facilitar la revisión de entregables y la resolución de dudas en demo.

---

## Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
print('Cargando dataset...')
df_raw = pd.read_excel(url)
print(f'Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
df_raw.head()

---
## Agente 1 – EDA

In [ ]:
def agente_eda(df):
    print('=' * 55)
    print('AGENTE 1 — EDA')
    print('=' * 55)

    # Estadísticas descriptivas
    print('\n── Estadísticas descriptivas ──')
    print(df.describe())

    # Nulos
    nulos = df.isnull().sum()
    pct   = (nulos / len(df) * 100).round(2)
    print('\n── Valores nulos ──')
    print(pd.DataFrame({'Nulos': nulos, 'Porcentaje': pct})[nulos > 0])

    # Histograma de UnitPrice (proxy de ventas antes de crear TotalPrice)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['UnitPrice'].clip(0, 20).hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title('Distribución de UnitPrice (recortado en 20£)')
    axes[0].set_xlabel('Precio unitario (£)')

    df['Quantity'].clip(0, 50).hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_title('Distribución de Quantity (recortado en 50)')
    axes[1].set_xlabel('Cantidad')
    plt.tight_layout(); plt.show()

    # Matriz de correlación
    plt.figure(figsize=(6, 4))
    numericas = df[['Quantity', 'UnitPrice']].dropna()
    sns.heatmap(numericas.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Matriz de correlación — variables numéricas crudas')
    plt.tight_layout(); plt.show()

    return {
        'shape': df.shape,
        'nulos': nulos[nulos > 0].to_dict(),
        'describe': df.describe()
    }

eda_output = agente_eda(df_raw)

### ✅ Respuestas esperadas — Agente 1

**¿Cuáles son las columnas con más nulos?**
- `CustomerID`: ~133,600 nulos (~24.9%). Corresponde a transacciones de clientes no registrados (ventas al mostrador o en efectivo sin cuenta).
- `Description`: ~1,454 nulos (~0.27%). Artículos sin descripción registrada, probablemente ajustes contables.

**¿Qué rango tienen Quantity y UnitPrice?**
- `Quantity`: min = -80,995 (devolución masiva), max = 80,995. La mayoría entre 1 y 12.
- `UnitPrice`: min = -11,062.06 (ajuste contable), max = 13,541.33. La mayoría entre £0.50 y £5.

**Anomalías detectadas:**
- Facturas con prefijo 'C' → cancelaciones con cantidades negativas.
- UnitPrice = 0 → posibles artículos de muestra o ajustes.
- Cantidad máxima de 80,995 → compra mayorista extrema, válida pero outlier.

**Correlaciones relevantes:**
- En el dataset crudo, Quantity y UnitPrice tienen correlación cercana a 0 o ligeramente negativa (artículos más baratos se compran en mayor cantidad). La correlación será más significativa en el dataset limpio con TotalPrice.

> 💡 **Para el instructor:** un equipo que solo ejecuta `df.describe()` sin interpretar los mínimos negativos no ha completado el EDA. El valor de -80,995 en Quantity debería disparar una pregunta inmediata en cualquier análisis real.

---
## Agente 2 – Limpieza de Datos

In [ ]:
def agente_limpieza(df):
    print('=' * 55)
    print('AGENTE 2 — LIMPIEZA DE DATOS')
    print('=' * 55)

    df_clean = df.copy()
    n_orig = len(df_clean)

    # 1. Eliminar Description nula (0.27% — no imputar: no hay valor lógico)
    df_clean = df_clean.dropna(subset=['Description'])

    # 2. Separar cancelaciones (InvoiceNo con 'C') para análisis aparte
    df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

    # 3. Eliminar precios y cantidades inválidos
    df_clean = df_clean[(df_clean['UnitPrice'] > 0) & (df_clean['Quantity'] > 0)]

    # 4. Eliminar duplicados exactos
    df_clean = df_clean.drop_duplicates()

    # 5. CustomerID nulo: conservar filas pero dejar como -1 (no eliminar porque
    #    representa 25% de las ventas reales; -1 indica 'cliente no registrado')
    df_clean['CustomerID'] = df_clean['CustomerID'].fillna(-1).astype(int)

    # 6. Crear variable objetivo
    df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

    # 7. Label Encoding de Country para el modelo
    le = LabelEncoder()
    df_clean['Country_enc'] = le.fit_transform(df_clean['Country'])

    # Reporte
    print(f'Registros originales:  {n_orig:>10,}')
    print(f'Registros limpios:     {len(df_clean):>10,}')
    print(f'Registros eliminados:  {n_orig - len(df_clean):>10,} ({(n_orig-len(df_clean))/n_orig*100:.1f}%)')
    print(f'\nVariable objetivo TotalPrice:')
    print(df_clean['TotalPrice'].describe())

    return df_clean, le

df_clean, label_encoder = agente_limpieza(df_raw)

### ✅ Respuestas esperadas — Agente 2

**¿Cuántos registros quedan después de la limpieza?**
- Resultado esperado: ~397,000–400,000 registros limpios (~73–74% del total original).
- Se eliminan: ~1,454 (Description nula) + ~9,288 (cancelaciones) + ~1,179 (precio ≤ 0) + duplicados.

**¿Qué variables categóricas se codifican?**
- `Country`: Label Encoding → un entero por país (38 valores). Se usa Label en lugar de One-Hot porque el modelo de árbol de decisión puede manejar enteros sin asumir orden; para regresión lineal One-Hot sería más correcto pero añade 38 columnas que complican la interpretación en un sprint de 3 horas.

**Decisión clave sobre CustomerID nulo:**
- No se eliminan esas filas (serían 25% de las ventas reales), se imputan con -1 como indicador de 'cliente no registrado'. Esta decisión debe documentarse en el reporte técnico.

> 💡 **Para el instructor:** si un equipo elimina directamente las filas con CustomerID nulo sin justificación, señalar que pierden ~133,000 transacciones válidas — casi un cuarto del dataset. Preguntar: ¿qué impacto tiene eso en la representatividad del modelo?

---
## Agente 3 – Modelo Predictivo

In [ ]:
def agente_modelo(df):
    print('=' * 55)
    print('AGENTE 3 — MODELO PREDICTIVO')
    print('=' * 55)

    # Features y variable objetivo
    features = ['Quantity', 'UnitPrice', 'Country_enc']
    X = df[features]
    y = df['TotalPrice']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f'Entrenamiento: {X_train.shape[0]:,} | Prueba: {X_test.shape[0]:,}')

    # Regresión Lineal
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    rmse_lr_train = np.sqrt(mean_squared_error(y_train, lr.predict(X_train)))
    rmse_lr_test  = np.sqrt(mean_squared_error(y_test,  lr.predict(X_test)))

    # Árbol de Decisión
    dt = DecisionTreeRegressor(max_depth=6, random_state=42)
    dt.fit(X_train, y_train)
    rmse_dt_train = np.sqrt(mean_squared_error(y_train, dt.predict(X_train)))
    rmse_dt_test  = np.sqrt(mean_squared_error(y_test,  dt.predict(X_test)))

    # Tabla comparativa
    resultados = pd.DataFrame([
        {'Modelo': 'Regresión Lineal',   'RMSE Entrenamiento': round(rmse_lr_train,2), 'RMSE Prueba': round(rmse_lr_test,2)},
        {'Modelo': 'Árbol de Decisión',  'RMSE Entrenamiento': round(rmse_dt_train,2), 'RMSE Prueba': round(rmse_dt_test,2)},
    ])
    print('\n── Tabla comparativa ──')
    print(resultados.to_string(index=False))

    # Selección del mejor modelo
    mejor = lr if rmse_lr_test < rmse_dt_test else dt
    nombre_mejor = 'Regresión Lineal' if rmse_lr_test < rmse_dt_test else 'Árbol de Decisión'
    print(f'\nModelo seleccionado: {nombre_mejor} (menor RMSE en prueba)')

    # Predicciones interpretables de ejemplo
    ejemplos = pd.DataFrame([
        {'Quantity': 6,  'UnitPrice': 2.55, 'Country_enc': 36},  # UK, articulo típico
        {'Quantity': 12, 'UnitPrice': 1.25, 'Country_enc': 36},  # UK, artículo económico
        {'Quantity': 3,  'UnitPrice': 8.50, 'Country_enc': 8},   # Alemania, artículo premium
    ])
    ejemplos['TotalPrice_Real']    = ejemplos['Quantity'] * ejemplos['UnitPrice']
    ejemplos['TotalPrice_Pred']    = mejor.predict(ejemplos[features]).round(2)
    print('\n── Predicciones de ejemplo ──')
    print(ejemplos)

    # Visualización real vs predicho
    y_pred_test = mejor.predict(X_test)
    plt.figure(figsize=(8, 4))
    plt.scatter(y_test[:500], y_pred_test[:500], alpha=0.3, color='steelblue')
    lim = max(y_test[:500].max(), y_pred_test[:500].max())
    plt.plot([0, lim], [0, lim], 'r--', label='Predicción perfecta')
    plt.xlabel('Valor real (£)'); plt.ylabel('Valor predicho (£)')
    plt.title(f'{nombre_mejor} — Real vs Predicho (primeras 500 muestras de prueba)')
    plt.legend(); plt.tight_layout(); plt.show()

    return mejor, resultados

modelo_final, tabla_modelos = agente_modelo(df_clean)

### ✅ Respuestas esperadas — Agente 3

**Resultados típicos esperados:**
| Modelo | RMSE Entrenamiento | RMSE Prueba |
|---|---|---|
| Regresión Lineal | ~20–25 £ | ~20–25 £ |
| Árbol de Decisión (depth=6) | ~15–18 £ | ~18–22 £ |

**¿Cuál modelo tuvo menor RMSE?**
- Típicamente el Árbol de Decisión, porque captura relaciones no lineales entre Quantity y TotalPrice (ej. pedidos grandes tienen descuentos implícitos).
- Si la diferencia es pequeña (<2£), la Regresión Lineal puede ser preferible por su interpretabilidad.

**¿Por qué comparar dos modelos en lugar de elegir uno directamente?**
- La comparación es el proceso científico mínimo: sin baseline (regresión lineal) no hay contexto para saber si el árbol 'realmente' mejora algo.
- En producción, elegir un modelo más complejo sin evidencia de mejora significativa introduce complejidad innecesaria de mantenimiento.
- Es el mismo principio que documentaron en M1: primero entender los datos, luego modelar.

**¿Las predicciones son interpretables?**
- Sí: 6 unidades × £2.55 → predicción esperada ~£15.30. Si el modelo predice £200, hay un error claro.
- La interpretabilidad es más importante en este contexto que el RMSE mínimo absoluto.

> 💡 **Para el instructor:** si un equipo reporta RMSE muy bajo (< 5£) en prueba, probablemente hay data leakage (usaron TotalPrice como feature sin darse cuenta) o el árbol tiene profundidad ilimitada y está sobreajustado. Preguntar: ¿cuál es el RMSE en entrenamiento vs prueba?

---
## Orquestador — Pipeline Completo

In [ ]:
def pipeline_completo(url_o_df):
    """
    Orquestador: ejecuta secuencialmente los 3 agentes
    y retorna un resumen ejecutivo del pipeline.
    """
    print('\n' + '█' * 55)
    print('  ORQUESTADOR — INICIO DEL PIPELINE')
    print('█' * 55 + '\n')

    # Cargar datos si se pasa una URL
    if isinstance(url_o_df, str):
        df = pd.read_excel(url_o_df)
    else:
        df = url_o_df.copy()

    # Agente 1
    eda_result = agente_eda(df)
    print(f'✅ Agente 1 completado — {eda_result["shape"][0]:,} registros analizados\n')

    # Agente 2
    df_limpio, le = agente_limpieza(df)
    print(f'✅ Agente 2 completado — {len(df_limpio):,} registros limpios\n')

    # Agente 3
    modelo, tabla = agente_modelo(df_limpio)
    print('✅ Agente 3 completado — modelo seleccionado\n')

    print('\n' + '█' * 55)
    print('  RESUMEN EJECUTIVO DEL PIPELINE')
    print('█' * 55)
    print(f'  Dataset original:  {eda_result["shape"][0]:>10,} registros')
    print(f'  Dataset limpio:    {len(df_limpio):>10,} registros')
    print(f'  Columnas con nulos: {list(eda_result["nulos"].keys())}')
    print(f'  Tabla de modelos:')
    print(tabla.to_string(index=False))
    print('█' * 55 + '\n')

    return {'eda': eda_result, 'df_limpio': df_limpio, 'modelo': modelo, 'tabla': tabla}

# Ejecutar el pipeline completo con el dataset ya cargado
resultado = pipeline_completo(df_raw)

### ✅ Respuestas esperadas — Preguntas de la demo

**1. ¿Por qué eligieron ese modelo como el mejor? ¿Solo por RMSE o hay otros factores?**
- RMSE es el criterio principal, pero también debe considerarse:
  - **Interpretabilidad**: la regresión lineal permite explicar el impacto de cada variable (coeficientes). El árbol de decisión es más difícil de explicar con profundidad > 3.
  - **Robustez a outliers**: el dataset tiene transacciones mayoristas muy grandes que sesgan la regresión lineal; el árbol de decisión las maneja mejor con `max_depth`.
  - **Tiempo de inferencia**: ambos son instantáneos para este volumen, no es factor diferenciador aquí.
  - **Mantenimiento**: un modelo más simple es más fácil de mantener si el dataset cambia.

**2. ¿Qué ocurre si el dataset tiene columnas con nombres diferentes?**
- El pipeline actual es frágil: falla si la columna se llama 'quantity' en minúsculas o 'unit_price' con guion bajo.
- Solución mínima: agregar al Agente 2 una función de normalización de nombres (`df.columns = df.columns.str.strip().str.lower()`) y un mapeo de columnas conocidas.
- Señal de un entregable avanzado: manejo de excepciones con mensajes claros de error en la interfaz.

**3. ¿Cómo adaptarían este pipeline para usarlo con sus estudiantes en el aula?**
- Respuesta mínima esperada: reemplazar el dataset de retail por uno del área de cada docente (ventas de cafetería, asistencia, calificaciones) y mantener la estructura de 3 agentes.
- Respuesta avanzada: enseñar a los estudiantes a construir cada agente como función independiente, mostrando explícitamente cómo la salida de uno es la entrada del siguiente — patrón pipeline aplicable en cualquier lenguaje.

**4. ¿Qué cambiarían con una semana más de desarrollo?**
- Respuestas válidas: agregar más features al modelo (mes, día de la semana extraídos de InvoiceDate), probar XGBoost como tercer modelo, agregar validación cruzada, autenticación en Streamlit, o despliegue en HuggingFace Spaces.
- Lo que NO es una respuesta válida: 'haríamos una red neuronal' sin justificación de por qué el dataset requiere esa complejidad.

---
### 💡 Rúbrica de evaluación rápida para la demo (referencia interna)

| Criterio | Mínimo (1) | Esperado (2) | Destacado (3) |
|---|---|---|---|
| Pipeline funcional | Solo notebook | Notebook + función pipeline | Streamlit con carga de CSV |
| Comparación de modelos | RMSE calculado | Tabla comparativa con análisis | Gráfica real vs predicho + justificación |
| Calidad del EDA | Solo `describe()` | Nulos + visualizaciones | Anomalías detectadas e interpretadas |
| Documentación | README básico | README + diagrama de arquitectura | Reporte técnico completo (5 pág) |
| Transferencia al aula | Mención genérica | Caso concreto de su asignatura | Plan de implementación con dataset propio |